计算Hessian矩阵
做一个假设，
View gradient norm less than 1e-3 as gradient equals to zero.
If minimum ratio is greater than 0.5 and gradient norm is less than 1e-3, then we assume that the model is at “local minima like”.
If minimum ratio is less than 0.5 and gradient norm is less than 1e-3, then we assume that the model is at “ saddle point”.

loss 为 e的平方
在MSE中，∂L / ∂W = Σ_output_nodes_j (∂L / ∂O_j) * (∂O_j / ∂W) （这里 O_j 是该层输出节点 j 的值）
则 ∂L / ∂W = B*A
则∂L / ∂W的导为∂（∂L / ∂W），
则其为2*∂L / ∂W*∂L / ∂W + （∂L / ∂W）平方（过小，忽略）
则H正比于∂L / ∂W*∂L / ∂W



In [ ]:
{
  "cells": [],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.9.0"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}

搁置再议

In [ ]:
student_id = 'your_student_id' # fill with your student ID

assert student_id != 'your_student_id', 'Please fill in your student_id before you start.'

In [ ]:

import numpy as np
from math import pi
from collections import defaultdict
from autograd_lib import autograd_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import warnings
warnings.filterwarnings("ignore")

Define the NN Model

In [ ]:
class MathRegressor(nn.Module):
    def __init__(self, num_hidden=128):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(1, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, 1)
        )

    def forward(self, x):
        x = self.regressor(x)
        return x

In [ ]:
#find the key from student_id
import re

key = student_id[-1]
if re.match('[0-9]',key) is not None:
    key = int(key)
else:
    key = ord(key)%10

In [ ]:
model = MathRegressor()
autograd_lib.register(model)

data = torch.load('data.pth')[key]
model.load_state_dict(data['model'])
train,target = data['data']

In [ ]:
#function to compute gradient norm
def compute_gradient_norm(model,criterion,train,target):
    model.train()
    model.zero_grad()
    output = model(train)
    loss = criterion(output,target)
    loss.backward()    #计算梯度
    
    grad = []
    for p in model.regressor.children():
        if isinstance(p,nn.Linear):
            param_norm = p.weight.grad.data.norm(2).item() #计算每一层的梯度范数
            grad.append(param_norm)
            
    grad_mean = np.mean(grad)   #取平均值
    
    return grad_mean

In [ ]:
#helper funtion
def save_activation(layer,A,_):
    activations[layer] = A
    
def compute_hess(layer,_,B):
    A= activations[layer]
    BA=torch.einsum('nl,ni->nli',B,A)  #计算外积
    
    hess[layer] += torch.einsum('nli,nkj->likj',BA,BA)  #累加Hessian矩阵
    

In [ ]:

# function to compute the minimum ratio
def compute_minimum_ratio(model, criterion, train, target):
    model.zero_grad()
    # compute Hessian matrix
    # save the gradient of each layer
    with autograd_lib.module_hook(save_activations):
        output = model(train)
        loss = criterion(output, target)

    # compute Hessian according to the gradient value stored in the previous step
    with autograd_lib.module_hook(compute_hess):
        autograd_lib.backward_hessian(output, loss='LeastSquares')

    layer_hess = list(hess.values())
    minimum_ratio = []

    # compute eigenvalues of the Hessian matrix
    for h in layer_hess:
        size = h.shape[0] * h.shape[1]
        h = h.reshape(size, size)
        h_eig = torch.symeig(h).eigenvalues # torch.symeig() returns eigenvalues and eigenvectors of a real symmetric matrix
        num_greater = torch.sum(h_eig > 0).item()
        minimum_ratio.append(num_greater / len(h_eig))

    ratio_mean = np.mean(minimum_ratio) # compute mean of minimum ratio

    return ratio_mean

In [ ]:
# the main function to compute gradient norm and minimum ratio
def main(model, train, target):
    criterion = nn.MSELoss()

    gradient_norm = compute_gradient_norm(model, criterion, train, target)
    minimum_ratio = compute_minimum_ratio(model, criterion, train, target)

    print('gradient norm: {}, minimum ratio: {}'.format(gradient_norm, minimum_ratio))

In [ ]:
if __name__ == '__main__':
    # fix random seed
    torch.manual_seed(0)

    # reset compute dictionaries
    activations = defaultdict(int)
    hess = defaultdict(float)

    # compute Hessian
    main(model, train, target)